In [90]:
from ultralytics import YOLO
import torch
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import os
import clip
import faiss
from matplotlib import colors
from matplotlib import patches
import threading
from segment_anything import SamPredictor, sam_model_registry
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as colors
import threading
import cv2
import pandas as pd
from PIL import Image
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [91]:
PREDICT_ARGS = {
    # Detection Settings
    'conf': 0.2,  # object confidence threshold for detection
    'iou': 0.3,  # intersection over union (IoU) threshold for NMS
    'imgsz': 640,  # image size as scalar or (h, w) list, i.e. (640, 480)
    'half': False,  # use half precision (FP16)
    'device': None,  # device to run on, i.e. cuda device=0/1/2/3 or device=cpu
    'max_det': 300,  # maximum number of detections per image
    'vid_stride': False,  # video frame-rate stride
    'stream_buffer': False,  # buffer all streaming frames (True) or return the most recent frame (False)
    'visualize': False,  # visualize model features
    'augment': False,  # apply image augmentation to prediction sources
    'agnostic_nms': False,  # class-agnostic NMS
    'classes': None,  # filter results by class, i.e. classes=0, or classes=[0,2,3]
    'retina_masks': False,  # use high-resolution segmentation masks
    'embed': None,  # return feature vectors/embeddings from given layers

    # Visualization Settings
    'show': False,  # show predicted images and videos if environment allows
    'save': False,  # save predicted images and videos
    'save_frames': False,  # save predicted individual video frames
    'save_txt': False,  # save results as .txt file
    'save_conf': False,  # save results with confidence scores
    'save_crop': False,  # save cropped images with results
    'show_labels': True,  # show prediction labels, i.e. 'person'
    'show_conf': True,  # show prediction confidence, i.e. '0.99'
    'show_boxes': True,  # show prediction boxes
    'line_width': None  # line width of the bounding boxes. Scaled to image size if None.
    }

In [92]:

index_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\correct_tool\faiss_index.index"
labels_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\correct_tool\labels.npy"
yolo_model = YOLO(r'C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\best.pt')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sam_checkpoint = r"C:\Users\user\Documents\code\korean_food_detection\tools\sam_vit_b_01ec64.pth"
model_type = "vit_b"

In [93]:
class FoodImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if os.path.isdir(class_path):
                for img_file in os.listdir(class_path):
                    img_path = os.path.join(class_path, img_file)
                    self.image_paths.append(img_path)
                    self.labels.append(class_folder)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [94]:
model, preprocess = clip.load("ViT-L/14@336px", device=device)

In [95]:
def load_faiss_index(index_path):
    index = faiss.read_index(index_path)
    print(f"FAISS index loaded from {index_path}")
    return index
def clip_transform(image):
    return preprocess(image)

In [96]:
def classify_image(model, faiss_index, image, transform, device, labels):
    """
    Classify an image using the CLIP model and FAISS index.

    Args:
    - model: The pre-trained CLIP model.
    - faiss_index: The FAISS index with image embeddings.
    - image: The input image to classify (PIL Image).
    - transform: The preprocessing transform used for the image (same as the one for training).
    - device: The device to run the model on (cuda or cpu).
    - labels: The list of labels corresponding to the FAISS index embeddings.

    Returns:
    - The predicted label for the given image or None if no match is found.
    """
    # Check if FAISS index is empty
    if faiss_index.ntotal == 0:
        print("FAISS index is empty, no embeddings available for search.")
        return None

    # Put model in evaluation mode
    model.eval()
    
    # Preprocess the input image
    with torch.no_grad():
        image = transform(image).unsqueeze(0).to(device)

        # Extract image features using the CLIP model
        image_features = model.encode_image(image).cpu().numpy()

        # Search in the FAISS index for the nearest neighbor
        distances, indices = faiss_index.search(image_features, k=1)  # k=1 to get the closest match

        # Check if we got a valid result
        if len(indices) == 0 or len(indices[0]) == 0:
            print("No match found in the FAISS index.")
            return None

        # Get the index of the closest match
        closest_idx = indices[0][0]

        # Check if the index is within the valid range of labels
        if closest_idx >= len(labels):
            print("Closest index is out of bounds for labels.")
            return None

        # Return the corresponding label
        predicted_label = labels[closest_idx]
        return predicted_label

In [97]:
def load_labels(labels_path):
    """
    Load labels from a saved numpy file.
    
    Args:
    - labels_path: Path to the saved labels file.
    
    Returns:
    - List of labels.
    """
    return np.load(labels_path, allow_pickle=True).tolist()

In [98]:
faiss_index = load_faiss_index(index_path)
labels = load_labels(labels_path)

FAISS index loaded from C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\correct_tool\faiss_index.index


In [99]:
def LoadSAMPredictor(sam_checkpoint, model_type, device='cuda', return_sam=False):
    """
    Load a Segment Anything Model (SAM) predictor model for semantic segmentation.

    Parameters:
    - sam_checkpoint (str): The path to the checkpoint file containing the SAM model's weights and configuration.
    - model_type (str): The SAM model type to use. It should be a key that corresponds to a model in the 'sam_model_registry'.
    - device (str, optional): The device to run the model on, either 'cuda' (GPU) or 'cpu' (CPU). Default is 'cuda'.

    Returns:
    - predictor (SamPredictor): An instance of the SAM predictor configured with the specified model type and loaded weights.
    """
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    sam.to(device=device)
    predictor = SamPredictor(sam)

    if return_sam :
        return predictor,sam
    else : 
        return predictor

In [100]:
sam_predictor = LoadSAMPredictor(sam_checkpoint, model_type, device='cuda', return_sam=False)

In [101]:
def load_calories_data(csv_path):
    """
    Load calories data from a CSV file.
    
    Parameters:
    -----------
    csv_path : str
        Path to the CSV file containing food names and calories
    
    Returns:
    --------
    dict
        A dictionary mapping food names to their calorie values
    """
    try:
        # calories_df = pd.read_csv(csv_path)
        calories_df = pd.read_excel(csv_path)
        # Assuming the CSV has columns 'food_name' and 'calories'
        return dict(zip(calories_df['food_name'], calories_df['calory_number']))
    except Exception as e:
        print(f"Error loading calories data: {e}")
        return {}

In [102]:
def run_sam_with_multiple_points(sam_predictor, input_point, bbox, all_masks, index):
    x1, y1, x2, y2 = bbox
    input_label = np.array([1])
    mask, _, _ = sam_predictor.predict(
        point_coords=np.array([input_point]),
        point_labels=input_label,
        box=np.array([x1, y1, x2, y2])[None, :],
        multimask_output=False,
    )
    all_masks[index] = mask[0]

In [103]:
def get_food_calories(food_name, calories_dict):
    """
    Retrieve calorie information for a given food name.
    
    Parameters:
    -----------
    food_name : str
        Name of the food
    calories_dict : dict
        Dictionary of food names and their calorie values
    
    Returns:
    --------
    int
        Number of calories for the food, or 0 if not found
    """
    # Try exact match first, then case-insensitive match
    calories = calories_dict.get(food_name, 
                                 calories_dict.get(food_name.lower(), 
                                 calories_dict.get(food_name.title(), 0)))
    return calories

In [147]:
def load_nutrition_data(csv_path):
    """
    Load nutrition data from a CSV file.
    
    Parameters:
    -----------
    csv_path : str
        Path to the CSV file containing food names and nutrition values
    
    Returns:
    --------
    dict
        A dictionary mapping food names to their nutritional values
    """
    try:
        nutrition_df = pd.read_excel(csv_path)
        # Create a dictionary with all nutrition columns
        return nutrition_df.set_index('food_name').to_dict(orient='index')
    except Exception as e:
        print(f"Error loading nutrition data: {e}")
        return {}

def get_food_nutrition(food_name, nutrition_dict):
    """
    Retrieve nutrition information for a given food name.
    
    Parameters:
    -----------
    food_name : str
        Name of the food
    nutrition_dict : dict
        Dictionary of food names and their nutritional values
    
    Returns:
    --------
    dict
        Nutritional values for the food, or default zeros if not found
    """
    # Try variations of food name matching
    nutrition_info = nutrition_dict.get(food_name, 
                                        nutrition_dict.get(food_name.lower(), 
                                        nutrition_dict.get(food_name.title(), 
                                        {'calories': 0, 'protein': 0, 'lipides': 0, 'glucides': 0})))
    return nutrition_info

# def plot_food_detection_with_nutrition(image_path, 
#                                        yolo_model, 
#                                        sam_predictor, 
#                                        model, 
#                                        faiss_index, 
#                                        clip_transform, 
#                                        device, 
#                                        labels, 
#                                        nutrition_csv_path):
#     """
#     Detect and classify food items with nutrition information.
    
#     Parameters:
#     -----------
#     image_path : str
#         Path to the input image
#     yolo_model : object
#         YOLO detection model
#     sam_predictor : object
#         Segment Anything Model predictor
#     model : object
#         CLIP model for classification
#     faiss_index : object
#         FAISS index for similarity search
#     clip_transform : object
#         Image transformation for CLIP
#     device : str
#         Computing device (cuda/cpu)
#     labels : list
#         List of food labels
#     nutrition_csv_path : str
#         Path to CSV file with nutrition information
#     """
#     # Load nutrition data
#     nutrition_dict = load_nutrition_data(nutrition_csv_path)
    
#     # Image preprocessing
#     image = Image.open(image_path).convert('RGB')
#     new_width = 512
#     aspect_ratio = new_width / image.width
#     new_height = int(image.height * aspect_ratio)
#     image = image.resize((new_width, new_height), Image.LANCZOS)
#     image_np = np.array(image)

#     # Detect objects with YOLO
#     detection_results = yolo_model(image, **PREDICT_ARGS)

#     # Predefined colormap for bounding boxes
#     colors_list = list(colors.CSS4_COLORS.keys())
#     np.random.seed(42)
#     random_colors = np.random.choice(colors_list, size=len(detection_results[0].boxes), replace=False)

#     # Set the image for SAM predictor
#     sam_predictor.set_image(image_np)

#     # Initialize storage for masks
#     all_masks = [None] * len(detection_results[0].boxes)

#     # Plot image with detection results
#     fig, ax = plt.subplots(1, figsize=(12, 8))
#     ax.imshow(image_np)

#     if len(detection_results) > 0:
#         detections = detection_results[0].boxes.xyxy
#         threads = []

#         for i, detection in enumerate(detections):
#             x1, y1, x2, y2 = map(int, detection[:4].tolist())

#             # Classify the cropped image using CLIP+FAISS
#             cropped_image = Image.fromarray(image_np[y1:y2, x1:x2])
#             predicted_label = classify_image(model, faiss_index, cropped_image, clip_transform, device, labels)

#             # Get nutrition for the detected food
#             nutrition_info = get_food_nutrition(predicted_label, nutrition_dict)

#             # Plot the bounding box from YOLO
#             color = random_colors[i]
#             rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, 
#                                      linewidth=2, edgecolor=color, facecolor='none')
#             ax.add_patch(rect)

#             # Prepare nutrition text
#             nutrition_text = (
#                 f'{predicted_label or "Unknown"}\n'
#                 f'Protein: {nutrition_info.get("protein", 0)}g\n'
#                 f'Lipides: {nutrition_info.get("lipides", 0)}g\n'
#                 f'Glucides: {nutrition_info.get("glucides", 0)}g'
#             )

#             # Display predicted label and nutrition info
#             ax.text(x1, y1 - 50, nutrition_text, 
#                     color='white', fontsize=9, 
#                     backgroundcolor=color, 
#                     verticalalignment='top')

#             # Generate SAM mask using the custom function
#             center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
#             input_point = np.array([center_x, center_y])
#             bbox = [x1, y1, x2, y2]
#             thread = threading.Thread(target=run_sam_with_multiple_points, 
#                                       args=(sam_predictor, input_point, bbox, all_masks, i))
#             thread.start()
#             threads.append(thread)

#         # Wait for all threads to finish
#         for thread in threads:
#             thread.join()

#         # Post-process and display masks for each detection
#         for i, mask in enumerate(all_masks):
#             if mask is not None:
#                 binary_mask = (mask > 0.5).astype(np.uint8)
#                 kernel = np.ones((5, 5), np.uint8)
#                 binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
#                 binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

#                 # Find contours and select the largest one
#                 contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#                 if contours:
#                     largest_contour = max(contours, key=cv2.contourArea)

#                     refined_mask = np.zeros_like(binary_mask)
#                     cv2.drawContours(refined_mask, [largest_contour], 0, 1, -1)

#                     # Plot the refined segmentation mask
#                     color_rgba = colors.to_rgba(random_colors[i], alpha=0.4)
#                     mask_overlay = np.zeros((*refined_mask.shape, 4), dtype=np.float32)
#                     mask_overlay[refined_mask == 1] = color_rgba
#                     ax.imshow(mask_overlay)
#     else:
#         print("No objects detected.")

#     # Show the image with bounding boxes, segmentation masks, labels, and nutrition info
#     plt.axis('off')
#     plt.tight_layout()
#     plt.show()



def plot_food_detection_with_nutrition(image_path, 
                                       yolo_model, 
                                       sam_predictor, 
                                       model, 
                                       faiss_index, 
                                       clip_transform, 
                                       device, 
                                       labels, 
                                       nutrition_csv_path, 
                                       output_folder):
    """
    Detect and classify food items with nutrition information, and save the result image to a specified folder.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image
    yolo_model : object
        YOLO detection model
    sam_predictor : object
        Segment Anything Model predictor
    model : object
        CLIP model for classification
    faiss_index : object
        FAISS index for similarity search
    clip_transform : object
        Image transformation for CLIP
    device : str
        Computing device (cuda/cpu)
    labels : list
        List of food labels
    nutrition_csv_path : str
        Path to CSV file with nutrition information
    output_folder : str
        Folder path to save the result image
    """
    # Load nutrition data
    nutrition_dict = load_nutrition_data(nutrition_csv_path)
    
    # Image preprocessing
    image = Image.open(image_path).convert('RGB')
    new_width = 512
    aspect_ratio = new_width / image.width
    new_height = int(image.height * aspect_ratio)
    image = image.resize((new_width, new_height), Image.LANCZOS)
    image_np = np.array(image)

    # Detect objects with YOLO
    detection_results = yolo_model(image, **PREDICT_ARGS)

    # Predefined colormap for bounding boxes
    colors_list = list(colors.CSS4_COLORS.keys())
    np.random.seed(42)
    random_colors = np.random.choice(colors_list, size=len(detection_results[0].boxes), replace=False)

    # Set the image for SAM predictor
    sam_predictor.set_image(image_np)

    # Initialize storage for masks
    all_masks = [None] * len(detection_results[0].boxes)

    # Plot image with detection results
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image_np)

    if len(detection_results) > 0:
        detections = detection_results[0].boxes.xyxy
        threads = []

        for i, detection in enumerate(detections):
            x1, y1, x2, y2 = map(int, detection[:4].tolist())

            # Classify the cropped image using CLIP+FAISS
            cropped_image = Image.fromarray(image_np[y1:y2, x1:x2])
            predicted_label = classify_image(model, faiss_index, cropped_image, clip_transform, device, labels)

            # Get nutrition for the detected food
            nutrition_info = get_food_nutrition(predicted_label, nutrition_dict)

            # Plot the bounding box from YOLO
            color = random_colors[i]
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, 
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

            # Prepare nutrition text
            nutrition_text = (
                f'{predicted_label or "Unknown"}\n'
                f'Protein: {nutrition_info.get("protein", 0)}g\n'
                f'Lipides: {nutrition_info.get("lipides", 0)}g\n'
                f'Glucides: {nutrition_info.get("glucides", 0)}g'
            )

            # Display predicted label and nutrition info
            ax.text(x1, y1 - 50, nutrition_text, 
                    color='white', fontsize=9, 
                    backgroundcolor=color, 
                    verticalalignment='top')

            # Generate SAM mask using the custom function
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            input_point = np.array([center_x, center_y])
            bbox = [x1, y1, x2, y2]
            thread = threading.Thread(target=run_sam_with_multiple_points, 
                                      args=(sam_predictor, input_point, bbox, all_masks, i))
            thread.start()
            threads.append(thread)

        # Wait for all threads to finish
        for thread in threads:
            thread.join()

        # Post-process and display masks for each detection
        for i, mask in enumerate(all_masks):
            if mask is not None:
                binary_mask = (mask > 0.5).astype(np.uint8)
                kernel = np.ones((5, 5), np.uint8)
                binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
                binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

                # Find contours and select the largest one
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    largest_contour = max(contours, key=cv2.contourArea)

                    refined_mask = np.zeros_like(binary_mask)
                    cv2.drawContours(refined_mask, [largest_contour], 0, 1, -1)

                    # Plot the refined segmentation mask
                    color_rgba = colors.to_rgba(random_colors[i], alpha=0.4)
                    mask_overlay = np.zeros((*refined_mask.shape, 4), dtype=np.float32)
                    mask_overlay[refined_mask == 1] = color_rgba
                    ax.imshow(mask_overlay)
    else:
        print("No objects detected.")

    # Show the image with bounding boxes, segmentation masks, labels, and nutrition info
    plt.axis('off')
    plt.tight_layout()

    # Save the result image
    output_filename = os.path.basename(image_path)
    output_filepath = os.path.join(output_folder, output_filename)
    plt.savefig(output_filepath, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.close()

    # Optionally, display the result as well
    plt.show()

# Example usage
nutrition_csv_path = r'C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\correct_tool\New Feuille de calcul Microsoft Excel.xlsx'
image_path = r'C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\KakaoTalk_20240926_134421785_11_jpg.rf.e51bf6b9c4572a0c5ba1c18ec71c9729.jpg'

output_folder = r'C:\Users\user\Documents\GitHub\Korean_Food_Detection\test\result_images'
plot_food_detection_with_nutrition(image_path, yolo_model, sam_predictor, model, faiss_index, clip_transform, device, labels, nutrition_csv_path, output_folder)


0: 480x640 4 foods, 39.0ms
Speed: 3.0ms preprocess, 39.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)


In [71]:
import yaml

import os
import glob
import yaml

import os
import yaml




In [83]:
def load_ground_truth_classification(yaml_file, dataset_type='test'):
    """
    Load ground truth labels for classification from a YOLO format dataset.
    
    Args:
        yaml_file (str): Path to the YAML file with dataset configuration
        dataset_type (str): Dataset split to use ('train', 'val', 'test')
    
    Returns:
        dict: Mapping of image paths to ground truth class names
    """
    # Load YAML file
    with open(yaml_file, 'r') as file:
        data = yaml.safe_load(file)
    
    # Get dataset paths
    dataset_path = Path(os.path.dirname(yaml_file))
    
    # Handle different yaml formats
    if isinstance(data.get(dataset_type), str):
        images_folder = Path(data[dataset_type])
    else:
        images_folder = Path(data['path']) / dataset_type / 'images' if 'path' in data else None
        
    if not images_folder or not images_folder.exists():
        raise ValueError(f"Could not find valid images folder for {dataset_type} dataset")
    
    labels_folder = images_folder.parent / 'labels' / images_folder.name if 'labels' in str(images_folder.parent) else images_folder.parent / 'labels'
    
    # Ensure we have class names
    if 'names' not in data:
        raise ValueError("No class names found in YAML file")
    id_to_label = data['names']
    
    ground_truth = {}
    
    # Iterate through image files
    for img_path in images_folder.glob('*.[jp][pn][gn]*'):
        # Get corresponding label file
        label_file = labels_folder / f"{img_path.stem}.txt"
        
        if label_file.exists():
            try:
                with open(label_file, 'r') as f:
                    label_ids = set()
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 1:  # Ensure there's at least one value (class id)
                            try:
                                class_id = int(float(parts[0]))  # Handle potential floating point values
                                if 0 <= class_id < len(id_to_label):
                                    label_ids.add(class_id)
                            except (ValueError, IndexError) as e:
                                print(f"Warning: Invalid label format in {label_file}: {e}")
                                continue
                    
                    # Convert IDs to names
                    valid_labels = [id_to_label[id] for id in label_ids]
                    if valid_labels:
                        ground_truth[str(img_path)] = valid_labels
            except Exception as e:
                print(f"Error processing {label_file}: {e}")
    
    if not ground_truth:
        print(f"Warning: No valid ground truth labels found in {labels_folder}")
    
    return ground_truth


In [84]:
def evaluate_classification(ground_truth, model, faiss_index, clip_transform, device, labels):
    """
    Evaluate classification accuracy with detailed error analysis.
    
    Args:
        ground_truth (dict): Mapping of image paths to ground truth labels
        model: CLIP model
        faiss_index: FAISS index for similarity search
        clip_transform: CLIP preprocessing transform
        device: Computation device
        labels (list): Class names in FAISS index
    
    Returns:
        tuple: (accuracy, incorrect_predictions, detailed_metrics)
    """
    correct = 0
    total = 0
    incorrect_predictions = []
    class_metrics = {label: {'correct': 0, 'total': 0} for label in set(labels)}
    
    for image_path, true_labels in ground_truth.items():
        try:
            # Validate true labels
            valid_true_labels = [label for label in true_labels if label in labels]
            if not valid_true_labels:
                print(f"Skipping image with invalid labels: {image_path}")
                continue
            
            # Load and process image
            image = Image.open(image_path).convert('RGB')
            processed_image = clip_transform(image).unsqueeze(0).to(device)
            
            # Get predictions
            with torch.no_grad():
                image_features = model.encode_image(processed_image)
                image_features = image_features.cpu().numpy()
                image_features = image_features / np.linalg.norm(image_features, axis=1, keepdims=True)
            
            # Get top predictions
            k = min(5, len(labels))
            distances, indices = faiss_index.search(image_features, k)
            
            # Convert distances to confidence scores (assuming L2 distance)
            # Normalize distances to [0, 1] range and convert to confidence
            max_distance = np.max(distances[0])
            min_distance = np.min(distances[0])
            if max_distance == min_distance:
                confidence_scores = np.ones_like(distances[0])
            else:
                confidence_scores = 1 - (distances[0] - min_distance) / (max_distance - min_distance)
            
            predicted_labels = [labels[idx] for idx in indices[0] if idx < len(labels)]
            
            # Update metrics
            is_correct = any(pred in valid_true_labels for pred in predicted_labels[:1])
            if is_correct:
                correct += 1
            else:
                incorrect_predictions.append({
                    'image_path': image_path,
                    'true_labels': valid_true_labels,
                    'predicted_labels': predicted_labels,
                    'confidence_scores': confidence_scores.tolist()
                })
            
            # Update per-class metrics
            for label in valid_true_labels:
                class_metrics[label]['total'] += 1
                if is_correct:
                    class_metrics[label]['correct'] += 1
            
            total += 1
            
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            continue
    
    # Calculate final metrics
    accuracy = correct / total if total > 0 else 0
    
    # Calculate per-class accuracies
    detailed_metrics = {
        'overall_accuracy': accuracy,
        'total_images': total,
        'correct_predictions': correct,
        'per_class_accuracy': {
            label: {'accuracy': metrics['correct'] / metrics['total'] if metrics['total'] > 0 else 0,
                   'total_samples': metrics['total']}
            for label, metrics in class_metrics.items()
            if metrics['total'] > 0
        }
    }
    
    return accuracy, incorrect_predictions, detailed_metrics

In [85]:
from pathlib import Path

In [86]:
# Load ground truth
yaml_file = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\data.yaml"
ground_truth = load_ground_truth_classification(yaml_file, dataset_type='test')

# Evaluate
accuracy, incorrect_preds, metrics = evaluate_classification(
    ground_truth, 
    model, 
    faiss_index, 
    clip_transform, 
    device, 
    labels
)

# Print results
print(f"Overall accuracy: {metrics['overall_accuracy']:.2%}")
print("\nPer-class accuracies:")
for class_name, stats in metrics['per_class_accuracy'].items():
    print(f"{class_name}: {stats['accuracy']:.2%} ({stats['total_samples']} samples)")

Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\2-a_jpg.rf.0a74b3da24bf4a78d09cf10c88932ccb.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\3-a_jpg.rf.56d4af291532aa2e7669a0b26ab4b7f9.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\3-c_jpg.rf.f372927e0aa2ff69676a715942681aae.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\4-a_jpg.rf.f5b3d98a261809d86da88968352b0e43.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\5-a_jpg.rf.9cfb74120fa3d767f79e830e4b20e570.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\5-f_jpg.rf.ed1a5f0d592b686523974a92c937fffa.jpg
Skipping image with in

In [87]:
import pandas as pd
from datetime import datetime
import csv
from pathlib import Path

def generate_evaluation_report(ground_truth, model, faiss_index, clip_transform, device, labels, output_dir=None):
    """
    Generate a comprehensive evaluation report and save it as CSV.
    
    Args:
        ground_truth (dict): Mapping of image paths to ground truth labels
        model: CLIP model
        faiss_index: FAISS index for similarity search
        clip_transform: CLIP preprocessing transform
        device: Computation device
        labels (list): Class names in FAISS index
        output_dir (str, optional): Directory to save the report. If None, saves in current directory
    
    Returns:
        tuple: (accuracy, report_path, detailed_metrics)
    """
    # Create results directory if it doesn't exist
    if output_dir is None:
        output_dir = Path.cwd() / "evaluation_reports"
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Generate timestamp for unique filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = output_dir / f"evaluation_report_{timestamp}.csv"
    
    # Initialize lists to store results
    results = []
    correct = 0
    total = 0
    class_metrics = {label: {'correct': 0, 'total': 0} for label in set(labels)}
    
    # Process each image
    for image_path, true_labels in ground_truth.items():
        try:
            # Validate true labels
            valid_true_labels = [label for label in true_labels if label in labels]
            if not valid_true_labels:
                print(f"Skipping image with invalid labels: {image_path}")
                continue
            
            # Load and process image
            image = Image.open(image_path).convert('RGB')
            processed_image = clip_transform(image).unsqueeze(0).to(device)
            
            # Get predictions
            with torch.no_grad():
                image_features = model.encode_image(processed_image)
                image_features = image_features.cpu().numpy()
                image_features = image_features / np.linalg.norm(image_features, axis=1, keepdims=True)
            
            # Get top predictions
            k = min(5, len(labels))
            distances, indices = faiss_index.search(image_features, k)
            
            # Calculate confidence scores
            max_distance = np.max(distances[0])
            min_distance = np.min(distances[0])
            if max_distance == min_distance:
                confidence_scores = np.ones_like(distances[0])
            else:
                confidence_scores = 1 - (distances[0] - min_distance) / (max_distance - min_distance)
            
            predicted_labels = [labels[idx] for idx in indices[0] if idx < len(labels)]
            
            # Check if prediction is correct
            is_correct = any(pred in valid_true_labels for pred in predicted_labels[:1])
            if is_correct:
                correct += 1
                
            # Update class metrics
            for label in valid_true_labels:
                class_metrics[label]['total'] += 1
                if is_correct:
                    class_metrics[label]['correct'] += 1
            
            # Create result entry
            result = {
                'image_path': image_path,
                'true_labels': ', '.join(valid_true_labels),
                'top_prediction': predicted_labels[0] if predicted_labels else 'N/A',
                'top_confidence': f"{confidence_scores[0]*100:.2f}%" if len(confidence_scores) > 0 else 'N/A',
                'all_predictions': ', '.join(f"{pred} ({conf*100:.2f}%)" 
                                          for pred, conf in zip(predicted_labels, confidence_scores)),
                'is_correct': is_correct,
                'error_type': '' if is_correct else 'Misclassification'
            }
            results.append(result)
            total += 1
            
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            results.append({
                'image_path': image_path,
                'true_labels': ', '.join(valid_true_labels),
                'top_prediction': 'ERROR',
                'top_confidence': 'N/A',
                'all_predictions': 'N/A',
                'is_correct': False,
                'error_type': f'Processing Error: {str(e)}'
            })
    
    # Calculate accuracy
    accuracy = correct / total if total > 0 else 0
    
    # Calculate per-class metrics
    class_performance = {
        label: {
            'accuracy': metrics['correct'] / metrics['total'] if metrics['total'] > 0 else 0,
            'total_samples': metrics['total']
        }
        for label, metrics in class_metrics.items()
        if metrics['total'] > 0
    }
    
    # Save results to CSV
    df = pd.DataFrame(results)
    df.to_csv(report_path, index=False)
    
    # Generate summary metrics
    summary_path = output_dir / f"evaluation_summary_{timestamp}.txt"
    with open(summary_path, 'w') as f:
        f.write(f"Evaluation Summary\n")
        f.write(f"=================\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"Overall Accuracy: {accuracy*100:.2f}%\n")
        f.write(f"Total Images: {total}\n")
        f.write(f"Correct Predictions: {correct}\n\n")
        
        f.write("Per-Class Performance\n")
        f.write("====================\n")
        for label, metrics in class_performance.items():
            f.write(f"{label}:\n")
            f.write(f"  Accuracy: {metrics['accuracy']*100:.2f}%\n")
            f.write(f"  Total Samples: {metrics['total_samples']}\n")
    
    detailed_metrics = {
        'overall_accuracy': accuracy,
        'total_images': total,
        'correct_predictions': correct,
        'per_class_accuracy': class_performance
    }
    
    print(f"\nEvaluation report saved to: {report_path}")
    print(f"Summary report saved to: {summary_path}")
    
    return accuracy, report_path, detailed_metrics

# Example usage:
def run_evaluation(yaml_file, model, faiss_index, clip_transform, device, labels, output_dir=None):
    """
    Run the complete evaluation pipeline and generate reports.
    
    Args:
        yaml_file (str): Path to the dataset YAML file
        model: CLIP model
        faiss_index: FAISS index
        clip_transform: CLIP transform
        device: Computation device
        labels: List of class names
        output_dir (str, optional): Output directory for reports
    """
    # Load ground truth
    ground_truth = load_ground_truth_classification(yaml_file, dataset_type='test')
    
    # Run evaluation and generate report
    accuracy, report_path, metrics = generate_evaluation_report(
        ground_truth,
        model,
        faiss_index,
        clip_transform,
        device,
        labels,
        output_dir
    )
    
    # Print summary
    print(f"\nEvaluation Complete!")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print(f"\nPer-class accuracies:")
    for class_name, stats in metrics['per_class_accuracy'].items():
        print(f"{class_name}: {stats['accuracy']*100:.2f}% ({stats['total_samples']} samples)")
    
    return report_path, metrics

In [89]:
# Run the evaluation
report_path, metrics = run_evaluation(
    yaml_file=yaml_file,
    model=model,
    faiss_index=faiss_index,
    clip_transform=clip_transform,
    device=device,
    labels=labels,
    output_dir=r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow"  # optional
)

# The CSV will contain all predictions and details
# The summary text file will contain the overview statistics

Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\2-a_jpg.rf.0a74b3da24bf4a78d09cf10c88932ccb.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\3-a_jpg.rf.56d4af291532aa2e7669a0b26ab4b7f9.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\3-c_jpg.rf.f372927e0aa2ff69676a715942681aae.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\4-a_jpg.rf.f5b3d98a261809d86da88968352b0e43.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\5-a_jpg.rf.9cfb74120fa3d767f79e830e4b20e570.jpg
Skipping image with invalid labels: C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_roboflow\test\images\5-f_jpg.rf.ed1a5f0d592b686523974a92c937fffa.jpg
Skipping image with in